# LangGraph `create_agent` POC — interactive walkthrough

This notebook is an interactive counterpart to `agentrun_langgraph_poc.py`. It uses the repository's real `langgraph_poc` package, existing contracts, and the actual middleware adapters that reuse `react_safety.py`.

**Default mode is offline and deterministic.** It uses an in-process scripted chat model and a fake read-only tool binding. It does not call an external model provider.

## What you will inspect

1. The plan and role assignments.
2. The `create_agent` Investigator and its three middleware instances.
3. Every outer `StateGraph` node update as the graph runs.
4. The final result, selected/completed step, and safety instrumentation.
5. The exact messages seen by the scripted model.

The notebook does not change the workflow's implementation; it imports and exercises the repository module. Run it from the `repository-intelligence` project root so `src/` and the package metadata resolve correctly.

## 0. Environment setup

From a terminal, activate your project's virtual environment and run:

```bash
pip install -e ".[poc]"
```

Then start Jupyter from the `repository-intelligence` directory, for example:

```bash
jupyter lab
```

The optional `poc` extra declares the project's supported LangGraph/LangChain dependencies. This notebook intentionally does **not** use `--live` or require an API key.

In [ ]:
from pathlib import Path
import json
from pprint import pprint

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "pyproject.toml").exists(), (
    "Run this notebook with the repository-intelligence directory as the working directory."
)

from repo_intelligence.langgraph_poc import (
    AGENT_RECURSION_LIMIT,
    GRAPH_RECURSION_LIMIT,
    build_create_agent_investigator,
    build_poc_graph,
    run_poc,
)

from repo_intelligence.multiagent.contracts import InvestigationReport
from repo_intelligence.tools.protocol import ToolResult

print("Project root:", PROJECT_ROOT.resolve())
print("Outer graph recursion limit:", GRAPH_RECURSION_LIMIT)
print("Investigator agent recursion limit:", AGENT_RECURSION_LIMIT)

## 1. Define the deterministic scenario

The real Planner adapter expects the existing planning contract. To keep this run offline, `_ScriptedPlanner` below returns a response shaped like the existing planner's structured output. The goals are representative of the question; the graph validates the four-step shape rather than inventing a plan itself.

In [ ]:
QUESTION = "How do the agents, tools, and MCP servers interact?"
REPO_SCOPE = "this repository"
COMMIT_SCOPE = "pinned snapshot"

PLAN_PAYLOAD = {
    "objective": QUESTION,
    "steps": [
        {"id": "P1", "goal": "Identify the agent components and responsibilities"},
        {"id": "P2", "goal": "Trace how tools are registered and dispatched"},
        {"id": "P3", "goal": "Map how MCP servers integrate with the agents"},
        {"id": "P4", "goal": "Synthesize the architecture diagram specification"},
    ],
}

REPORT_ARGS = {
    "investigation_id": "inv-poc-demo",
    "question": QUESTION,
    "repository_scope": REPO_SCOPE,
    "commit_scope": COMMIT_SCOPE,
    "findings": [
        {
            "statement": "Agents dispatch read-only tools through a registry",
            "evidence_refs": [{
                "path": "src/repo_intelligence/tool_dispatcher.py",
                "line_start": 1,
                "line_end": 5,
            }],
            "support": "SUPPORTED",
        }
    ],
    "evidence_refs": [{
        "path": "src/repo_intelligence/tool_dispatcher.py",
        "line_start": 1,
        "line_end": 5,
    }],
    "open_questions": [],
    "uncertainties": [],
    "investigated_sources": ["src/repo_intelligence/tool_dispatcher.py"],
    "status": "complete",
}

pprint(PLAN_PAYLOAD)

## 2. Build the offline chat model

`create_agent` still uses a real LangChain agent runtime. Only the model response is scripted: the first response calls the fake `read_file` tool; the second returns the existing `InvestigationReport` structured-output tool call.

This allows us to exercise the agent and middleware without a provider call. It does **not** evaluate the quality of a live model's planning or investigation.

In [ ]:
from typing import Any, List, Optional

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from pydantic import Field

class SpyChatModel(BaseChatModel):
    """Scripted test model that records every model request."""
    queue: List[BaseMessage] = Field(default_factory=list)
    calls: List[dict] = Field(default_factory=list)

    @property
    def _llm_type(self) -> str:
        return "spy"

    def bind_tools(self, tools: Any, **kwargs: Any) -> Any:
        # The model returns the scripted tool calls below.
        return self

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Any = None,
        **kwargs: Any,
    ) -> ChatResult:
        self.calls.append({"messages": list(messages)})
        if not self.queue:
            raise AssertionError("The scripted model response queue is exhausted")
        return ChatResult(generations=[ChatGeneration(message=self.queue.pop(0))])

spy_model = SpyChatModel(queue=[
    AIMessage(content="", tool_calls=[{
        "name": "read_file",
        "args": {"path": "src/tool_dispatcher.py"},
        "id": "call-1",
        "type": "tool_call",
    }]),
    AIMessage(content="", tool_calls=[{
        "name": "InvestigationReport",
        "args": dict(REPORT_ARGS),
        "id": "call-2",
        "type": "tool_call",
    }]),
])

print("Scripted model ready; queued responses:", len(spy_model.queue))

## 3. Bind a fake read-only tool

The adapter expects a callable matching the project's existing tool protocol: one argument mapping in, one `ToolResult` out. We record calls so we can verify that only the expected tool was executed.

In [ ]:
tool_calls = []

def fake_read_file(arguments: Any) -> ToolResult:
    arguments = dict(arguments or {})
    tool_calls.append(arguments)
    return ToolResult(
        tool="read_file",
        success=True,
        data={
            "path": arguments.get("path", ""),
            "content": "dispatch: registry[name](arguments)",
        },
        source=None,
    )

bindings = {"read_file": fake_read_file}
print("Available offline bindings:", list(bindings))

## 4. Construct the Investigator with `create_agent`

`build_create_agent_investigator` is the repository's real factory. It wraps the bound project-style tool as a LangChain tool and constructs the agent with three middleware components:

- `RequestBudgetMiddleware`: checks every outgoing model request using the existing request-budget helpers.
- `ExecutionLimitMiddleware`: adapts `ReActLoopGuard` to model/tool hooks.
- `ObservationBoundingMiddleware`: bounds tool output before it enters the next model context.

The ordering matters: budget checks run first so a rejected request does not consume a model-call allowance.

In [ ]:
investigator = build_create_agent_investigator(
    model=spy_model,
    bindings=bindings,
)

print("Agent implementation:", type(investigator.agent).__name__)
print("Tools exposed to the agent:", investigator.tool_names)
print("Budget middleware:", type(investigator.budget).__name__)
print("Execution middleware:", type(investigator.execution).__name__)
print("Observation middleware:", type(investigator.observation).__name__)

assert investigator.tool_names == ["read_file"]

## 5. Construct the outer `StateGraph`

The compiled graph has explicit nodes for planning, routing, selecting P1, executing the Investigator, receiving its report, and returning the final response. We inspect the Mermaid representation where supported by the installed LangGraph version.

In [ ]:
class ScriptedPlanner:
    def __init__(self):
        self.requests = []

    def complete(self, request):
        from repo_intelligence.llm.models import LLMResponse, Message
        self.requests.append(request)
        return LLMResponse(
            message=Message(role="assistant", content=""),
            structured_output=dict(PLAN_PAYLOAD),
        )

planner = ScriptedPlanner()
graph = build_poc_graph(planner_client=planner, investigator=investigator)

print("Compiled graph type:", type(graph).__name__)
try:
    display(graph.get_graph().draw_mermaid())
except Exception as exc:
    print("Mermaid rendering unavailable in this version:", repr(exc))

## 6. Run the graph one node at a time (streaming updates)

This is the main learning cell. `stream_mode="updates"` yields each node's state update as it completes. We merge those updates into a local snapshot so you can inspect the evolving state.

Expected successful path:

`planner → query_router → orchestrator_select → investigator → orchestrator_receive → final_response`

The graph plans and routes all four steps but executes only P1.

In [ ]:
initial_state = {
    "question": QUESTION,
    "repository_scope": REPO_SCOPE,
    "commit_scope": COMMIT_SCOPE,
}

state_snapshot = dict(initial_state)
node_updates = []

for update in graph.stream(
    initial_state,
    config={"recursion_limit": GRAPH_RECURSION_LIMIT},
    stream_mode="updates",
):
    node_updates.append(update)
    print("\n--- node update ---")
    pprint(update)
    # Each update maps a node name to that node's partial state update.
    for node_name, partial_state in update.items():
        if isinstance(partial_state, dict):
            state_snapshot.update(partial_state)

print("\n--- accumulated state keys ---")
print(sorted(state_snapshot.keys()))

## 7. Inspect the plan and router decisions

These are the validated existing project models, not notebook-specific replacement contracts.

In [ ]:
plan = state_snapshot.get("plan")
assignments = state_snapshot.get("role_assignments", [])

print("Plan objective:", getattr(plan, "objective", None))
print("Plan steps:")
for step in getattr(plan, "steps", []):
    print(f"  {step.id}: {step.goal}")

print("\nRole assignments:")
for assignment in assignments:
    print(f"  {assignment.step_id} -> {assignment.role}")

## 8. Inspect the Orchestrator decision and final report

The first Orchestrator node selects P1 and constructs the existing `TaskPayload`. The second Orchestrator node receives and validates the `InvestigationReport` before marking P1 completed.

In [ ]:
print("Selected step:", state_snapshot.get("selected_step_id"))
print("Selected task:")
pprint(state_snapshot.get("selected_task"))

final = state_snapshot.get("final", {})
print("\nFinal status:", final.get("status"))
print("Failure:", final.get("failure"))
print("Error:", final.get("error"))
print("Completed step IDs:", final.get("completed_step_ids"))

report = final.get("report")
if report is not None:
    print("Report contract:", type(report).__name__)
    print("Report ID:", report.investigation_id)
    print("Report status:", report.status)
    print("Findings:")
    for finding in report.findings:
        print(f"  [{finding.support}] {finding.statement}")

## 9. Inspect safety instrumentation

The middleware objects expose records and a guard snapshot specifically to make behavior testable. Token values are estimates where the provider does not supply usage metadata; do not interpret them as exact provider token counts.

In [ ]:
print("Budget records:")
pprint(investigator.budget.records)

print("\nObservation bounding records:")
pprint(investigator.observation.records)

print("\nReAct execution guard snapshot:")
pprint(investigator.execution.guard.snapshot())

print("\nTool calls:")
pprint(tool_calls)

print("\nModel call count:", len(spy_model.calls))
print("Model queue remaining:", len(spy_model.queue))

## 10. Inspect the exact model inputs

This lets you see the messages that were actually passed to the scripted model at each turn. The observation middleware record and the next request can be compared to understand where bounding happens.

In [ ]:
for call_index, call in enumerate(spy_model.calls, start=1):
    print(f"\n===== model call {call_index} =====")
    for message_index, message in enumerate(call["messages"], start=1):
        print(f"[{message_index}] {type(message).__name__}")
        content = getattr(message, "content", "")
        if isinstance(content, str):
            print(content[:1500])
        else:
            pprint(content)
        tool_calls_in_message = getattr(message, "tool_calls", None)
        if tool_calls_in_message:
            print("tool calls:")
            pprint(tool_calls_in_message)

## 11. Assertions: validate the POC behavior

These checks mirror the essential success-path assertions from the module-level test flow. If one fails, inspect the state and instrumentation above before continuing.

In [ ]:
assert final.get("status") == "SUCCESS", final
assert [step.id for step in plan.steps] == ["P1", "P2", "P3", "P4"]
assert [(item.step_id, item.role) for item in assignments] == [
    ("P1", "INVESTIGATOR"),
    ("P2", "INVESTIGATOR"),
    ("P3", "INVESTIGATOR"),
    ("P4", "ANALYST"),
]
assert state_snapshot.get("selected_step_id") == "P1"
assert final.get("completed_step_ids") == ["P1"]
assert isinstance(report, InvestigationReport)
assert len(tool_calls) == 1
assert len(spy_model.calls) == 2

print("PASS: four steps planned; routing validated; P1 only executed; report contract validated.")

## 12. What this offline run proves — and what it does not

**It demonstrates:** the actual graph nodes and routing, `create_agent` construction, middleware hooks being exercised by a scripted model/tool loop, structured report validation, and observable state updates.

**It does not prove:** that a live model will produce a good plan/report, that provider token usage matches estimates, or that the full repository test suite passes. For regression coverage, also run:

```bash
python -m pytest tests/test_langgraph_poc.py -q
```

To run the original CLI-equivalent behavior without the notebook's intermediate inspection, use:

```bash
python agentrun_langgraph_poc.py --verbose
```

Do not enable live mode until the offline flow and focused tests pass.